# All-files evaluation (Path B): realistic interictal load

The main experiments drew interictal windows only from the interictal segments
of the **seizure-containing** recordings. This notebook re-extracts PDC VAR(20)
features from **every** CHB-MIT recording of each patient, including the
seizure-free files, so that:

* the model sees the full diversity of interictal activity (better
  characterisation of the "not-preictal" class), and
* the false-alarm rate is measured against the realistic interictal load a
  deployed system would face.

**Cost warning.** This processes all ~660 EDF files of the 21 retained patients
(VAR(20) fit per 20-s window). The first run may take **hours**; it caches per
patient to `cache_pdc_var20_allfiles/` and is resumable (re-running
skips patients already cached). Reduce to a subset first if you want a quick
sanity check (`patients = patients[:3]`).

**What changes downstream.** With the full interictal load the per-patient 5x
cap now binds for everyone, so prevalence becomes ~0.167 (uniform) instead of
~0.34; AUC is unaffected, AUC-PR/Skill are re-based, and FPR/h reflects true
deployment. The notebook prints an all-files vs seizure-files comparison so the
impact is explicit.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

import os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# [path set by bootstrap] CODE   = Path(r"<repo>/Code")
# [path set by bootstrap] CODEV2 = Path(r"<repo>/this repository")
OUT    = CODEV2/"outputs"; OUT.mkdir(exist_ok=True)
# sys.path.insert(0, str(CODE)); sys.path.insert(0, str(CODEV2/"src"))
from config import DATA_ROOT, EXCLUDED_PATIENTS, INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS, FS
# NOTE: feature extraction is done by scripts/extract_allfiles.py (needs mne).
# This notebook only loads the cache + runs LOPO, so it does not import mne.
from seizure_metrics import generate_alarms, event_sensitivity, false_alarms_per_hour, infer_seizure_groups
from seeds import set_global_seed
set_global_seed(42); SEED=42; PDC_ORDER=20; STEP_SEC=10; K,M,REFR=5,12,180
BANDS=[("delta",0.5,4.0),("theta",4.0,8.0),("alpha",8.0,13.0),("beta",13.0,30.0)]; BAND_NAMES=[b[0] for b in BANDS]
ALL_CACHE=CODE/"cache_pdc_var20_allfiles"; ALL_CACHE.mkdir(exist_ok=True)
patients=sorted(p for p in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT,p)) and p.startswith("chb") and p not in EXCLUDED_PATIENTS)
print(len(patients),"patients")


21 patients


## PDC feature functions (identical to V6 / DTF notebooks)

## Load the all-files features

**Run the extraction in a terminal FIRST** (it is slow and would crash a
notebook kernel):

```
cd "scripts"
python extract_allfiles.py            # all patients (hours; resumable)
python extract_allfiles.py chb01 chb02   # quick 2-patient test
```

That script writes `cache_pdc_var20_allfiles/<pid>/`. This cell only
**loads** that cache into memory (fast, ~a few hundred MB); it never extracts,
so it cannot hang the kernel. If a patient is missing it is reported and
skipped &#8212; finish the extraction for it and re-run this cell.

In [2]:

ALL_CACHE = CODE / "cache_pdc_var20_allfiles"
DATA={}; missing=[]
for pid in patients:
    fp=ALL_CACHE/pid/"features.npy"; lp=ALL_CACHE/pid/"labels.npy"
    if not (fp.exists() and lp.exists()):
        missing.append(pid); continue
    X=np.load(fp); y=np.load(lp)
    if (y==1).sum()>0: DATA[pid]=(X,y)
    print(f"  {pid}: {X.shape}  pre={int((y==1).sum())} int={int((y==0).sum())} ratio 1:{int((y==0).sum())/max(int((y==1).sum()),1):.0f}")
if missing:
    print("\nNOT YET CACHED (run extract_allfiles.py for these):", ", ".join(missing))
patients=[p for p in patients if p in DATA]
assert patients, "No all-files cache found. Run scripts/extract_allfiles.py first."
tot_pre=sum(int((y==1).sum()) for _,y in DATA.values()); tot_int=sum(int((y==0).sum()) for _,y in DATA.values())
print(f"\nALL-FILES totals: preictal={tot_pre} interictal={tot_int} ratio 1:{tot_int/tot_pre:.0f} (was ~1:2.8 seizure-files-only)")


  chb01: (10763, 268)  pre=296 int=10467 ratio 1:35
  chb02: (11131, 268)  pre=296 int=10835 ratio 1:37
  chb03: (11352, 268)  pre=444 int=10908 ratio 1:25
  chb04: (54164, 268)  pre=296 int=53868 ratio 1:182
  chb05: (11154, 268)  pre=444 int=10710 ratio 1:24
  chb06: (18766, 268)  pre=1037 int=17729 ratio 1:17
  chb07: (22675, 268)  pre=445 int=22230 ratio 1:50
  chb08: (4708, 268)  pre=741 int=3967 ratio 1:5
  chb09: (22527, 268)  pre=592 int=21935 ratio 1:37
  chb10: (14335, 268)  pre=888 int=13447 ratio 1:15
  chb13: (8576, 268)  pre=444 int=8132 ratio 1:18
  chb14: (6836, 268)  pre=774 int=6062 ratio 1:8
  chb15: (6798, 268)  pre=320 int=6478 ratio 1:20
  chb16: (4046, 268)  pre=347 int=3699 ratio 1:11
  chb17: (6592, 268)  pre=444 int=6148 ratio 1:14
  chb18: (11351, 268)  pre=592 int=10759 ratio 1:18
  chb19: (9989, 268)  pre=296 int=9693 ratio 1:33
  chb20: (7689, 268)  pre=369 int=7320 ratio 1:20
  chb22: (9617, 268)  pre=296 int=9321 ratio 1:31
  chb23: (6888, 268)  pre=608 

## LOPO on the all-files data (flagship PDC-SVM and LR), event-level metrics

In [3]:

def cap_balance(X,y,seed):
    r=np.random.default_rng(seed); npre=int((y==1).sum()); ii=np.where(y==0)[0]
    keep=min(len(ii),INTERICTAL_MULTIPLIER*npre,MAX_INTERICTAL_ABS)
    sel=np.sort(np.concatenate([np.where(y==1)[0],r.choice(ii,keep,replace=False)])); return X[sel],y[sel]
def mk(name):
    c=LogisticRegression(max_iter=400,class_weight="balanced",C=0.1,random_state=SEED) if name=="LR" else SVC(kernel="rbf",C=0.1,class_weight="balanced",probability=False,random_state=SEED)
    return Pipeline([("s",StandardScaler()),("c",c)])
def scr(m,X):
    c=m.named_steps["c"]
    if hasattr(c,"predict_proba") and getattr(c,"probability",True): return m.predict_proba(X)[:,1]
    return 1/(1+np.exp(-m.decision_function(X)))
def run(name,sub=None):
    rows=[]
    for i,test in enumerate(patients):
        Xtr,ytr=[],[]
        for p in patients:
            if p==test: continue
            Xc,yc=cap_balance(*DATA[p],SEED+i); Xtr.append(Xc); ytr.append(yc)
        Xtr=np.vstack(Xtr); ytr=np.concatenate(ytr)
        if sub and len(ytr)>sub:
            r=np.random.default_rng(SEED+i); idx=r.choice(len(ytr),sub,replace=False); Xtr,ytr=Xtr[idx],ytr[idx]
        m=mk(name).fit(Xtr,ytr); X,y=DATA[test]
        if len(np.unique(y))<2: continue
        pr=scr(m,X); prev=float((y==1).mean())
        al=generate_alarms(pr,0.5,K,M,REFR)
        rows.append({"patient":test,"auc":roc_auc_score(y,pr),"auc_pr":average_precision_score(y,pr),
                     "prevalence":prev,"skill":(average_precision_score(y,pr)-prev)/(1-prev),
                     "event_sens":event_sensitivity(al,infer_seizure_groups(y)),"fpr_h":false_alarms_per_hour(al,y,STEP_SEC)})
    return pd.DataFrame(rows)
res={}
for nm,sub in [("LR",None),("SVM",20000)]:
    d=run(nm,sub); res[nm]=d; d.to_csv(OUT/f"allfiles_lopo_{nm}.csv",index=False)
    print(f"ALL-FILES {nm}: AUC={d.auc.mean():.3f} AUC-PR={d.auc_pr.mean():.3f} Skill={d.skill.mean():+.3f} prev={d.prevalence.mean():.3f} eventSens={d.event_sens.mean():.2f} FPR/h={d.fpr_h.mean():.2f}")


ALL-FILES LR: AUC=0.544 AUC-PR=0.087 Skill=+0.032 prev=0.059 eventSens=0.79 FPR/h=1.52
ALL-FILES SVM: AUC=0.520 AUC-PR=0.086 Skill=+0.031 prev=0.059 eventSens=0.67 FPR/h=1.37


## All-files vs seizure-files-only comparison

In [4]:

print(f"{'metric':12s} {'seizure-files (orig)':22s} {'all-files (this run)':22s}")
orig={"AUC":0.566,"AUC-PR":0.402,"Skill":0.091,"prevalence":0.34}
sv=res["SVM"]
allf={"AUC":sv.auc.mean(),"AUC-PR":sv.auc_pr.mean(),"Skill":sv.skill.mean(),"prevalence":sv.prevalence.mean()}
for k in ["AUC","AUC-PR","Skill","prevalence"]:
    print(f"{k:12s} {orig[k]:<22.3f} {allf[k]:<22.3f}")
print(f"\nEvent sensitivity (SVM, thr 0.5): {sv.event_sens.mean():.2f} | FPR/h: {sv.fpr_h.mean():.2f} (now on realistic interictal load)")
print("\nInterpretation: AUC should be roughly unchanged (prevalence-independent).")
print("AUC-PR/Skill re-based to the lower prevalence. FPR/h is now deployment-realistic.")
print("Send these numbers back and they will be wired into the Data and Results sections.")


metric       seizure-files (orig)   all-files (this run)  
AUC          0.566                  0.520                 
AUC-PR       0.402                  0.086                 
Skill        0.091                  0.031                 
prevalence   0.340                  0.059                 

Event sensitivity (SVM, thr 0.5): 0.67 | FPR/h: 1.37 (now on realistic interictal load)

Interpretation: AUC should be roughly unchanged (prevalence-independent).
AUC-PR/Skill re-based to the lower prevalence. FPR/h is now deployment-realistic.
Send these numbers back and they will be wired into the Data and Results sections.
